## Generate ESM Embeddings — Pilot (65k Family Centroids)

Lead     : `Dennis Zhu / Pixeldom04`

Issue    : [Github Issue #32](https://github.com/petadex/igem-toronto/issues/32) - Generate ESM Embeddings (Pilot 65k family centroids)

Start    : `2026-03-17`

Complete : `2026-03-20`

Files    : `~/resources/20260317_issue32_esm_embeddings/` — local working directory (scripts, logs, intermediate outputs)

S3 files : `s3://petadex/esm_embeddings/` — remote archive (final outputs, shared with team)

---

### Data Accessed: 2026-03-17
```bash
aws s3 cp s3://petadex/esm_embeddings/family_embeddings.npz ./
aws s3 cp s3://petadex/esm_embeddings/family_representatives.csv ./
```

## Introduction

PETadex organizes viral proteins into ~64,730 sequence families, each represented by a centroid sequence. To support downstream visualization and analysis, this pilot generates dense vector embeddings for all family centroids using ESM2 (`esm2_t30_150M_UR50D`), a pretrained protein language model. Mean-pooled residue-level representations from the final transformer layer are used as fixed-length, 640-dimensional embeddings per sequence. These embeddings serve as the foundation for the [ESM Landscape](https://petadex.net/atlas/) — a UMAP projection that places all 65k families in a shared 2D space for visual exploration.

## Objectives

1. Export all 64,730 nr family centroid sequences from petadex as a CSV
2. Run ESM2 (`esm2_t30_150M_UR50D`) on all centroids to generate mean-pooled 640-dimensional embeddings
3. Save embeddings as a compressed `.npz` archive and upload to S3
4. Make embeddings available for downstream landscape visualization (Issue #13)

---

## Materials and Methods

### System Initialization

Embeddings were generated on an EC2 `g4dn.xlarge` instance (NVIDIA T4 GPU, 16 GB VRAM) running Amazon Linux 2, using the `fair-esm` package.

```bash
pip install fair-esm
python -c "import esm; print(esm.__version__)"
# fair-esm 2.0.0

python -c "import torch; print(torch.__version__)"
# 2.x (CUDA enabled)
```

### Data Initialization

Family representative sequences were queried from the petadex database and saved as `family_representatives.csv` (columns: `family_id`, `sequence`).

```bash
# Accessed: 2026-03-17
# 64,730 nr family centroid sequences
```

### Embedding

Embeddings were generated using [`embed_families.py`](../resources/20260317_issue32_esm_embeddings/nr-families-embeddings/embed_families.py), run via [`run_embed.sh`](../resources/20260317_issue32_esm_embeddings/nr-families-embeddings/run_embed.sh).

The script:
- Loads sequences from `family_representatives.csv`, truncating to 1022 residues (ESM2 hard limit)
- Remaps non-standard IUPAC amino acid codes to ESM2-safe equivalents
- Runs `esm2_t30_150M_UR50D` in batches of 64 on GPU
- Extracts layer-30 residue representations and mean-pools across residues (excluding BOS/EOS tokens)
- Saves a checkpoint every 50 batches to allow resuming on failure
- Outputs a compressed `.npz` file with `family_ids` and `embeddings` arrays

```bash
# Run on EC2 g4dn.xlarge
nohup bash run_embed.sh > embed.log 2>&1 &
```

Results were uploaded to S3 on completion:

```bash
# Uploaded: 2026-03-20
aws s3 cp family_embeddings.npz s3://petadex/esm_embeddings/family_embeddings.npz
aws s3 cp family_representatives.csv s3://petadex/esm_embeddings/family_representatives.csv
```

## Results & Discussion

ESM2 embeddings were generated successfully for all 64,730 nr family centroids.

**Output**: `family_embeddings.npz` — shape `(64730, 640)`, dtype `float32`

**Runtime**: ~57 minutes on a single T4 GPU (15:04 → 16:01 UTC, batch size 64)

**Coverage**: 64,730 / 64,730 sequences embedded (100%)

The embeddings were used by Issue #13 to generate the ESM Landscape — a UMAP projection of all family centroids into 2D space, now live at [petadex.net/atlas/](https://petadex.net/atlas/).

**Follow-up questions**:
- Should embeddings be regenerated with a larger ESM2 model (e.g. `esm2_t33_650M_UR50D`) for higher-quality representations?
- For families with sequences >1022 residues, should a sliding-window or chunked embedding strategy be used instead of hard truncation?
- Can these embeddings support downstream tasks beyond visualization (e.g. family-level similarity search, clustering, or ML feature inputs)?